# Muria: AfriqueGemma-4B Adapter Merge & GGUF Quantization

This dedicated export notebook pulls the trained LoRA adapter from Hugging Face (`fallback-ai/Muria-Afrique-Gemma-4B-LoRA`), merges it with the base model `McGill-NLP/AfriqueGemma-4B`, converts the merged weights to **GGUF (Q4_K_M)** via `llama.cpp`, verifies output with a test prompt, and publishes the GGUF models directly to Hugging Face (`fallback-ai/Muria-Afrique-Gemma-4B`).

## Environment Setup & Dependency Installation

In [1]:

!pip install -q -U torchao transformers accelerate peft huggingface_hub gguf sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 80.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.2 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are i

## Authentication & Configuration

In [3]:
import os
import gc
import torch
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

print("Authenticating with Hugging Face...")
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN2")
except Exception:
    hf_token = os.environ.get("HF_TOKEN2") or os.environ.get("HF_TOKEN")

login(token=hf_token)
api = HfApi(token=hf_token)

# Repository and Directory Configuration
BASE_MODEL = "McGill-NLP/AfriqueGemma-4B"
LORA_REPO = "fallback-ai/Muria-Afrique-Gemma-4B-LoRA"
TARGET_REPO = "fallback-ai/Muria-Afrique-Gemma-4B"

WORKING_DIR = "/kaggle/working"
MERGED_DIR = f"{WORKING_DIR}/muria-afriquegemma-4b-faw-merged"
GGUF_DIR = f"{WORKING_DIR}/muria_gguf_export"

for d in [MERGED_DIR, GGUF_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Base Model:  {BASE_MODEL}")
print(f"LoRA Source: {LORA_REPO}")
print(f"Target Repo: {TARGET_REPO}")


Authenticating with Hugging Face...
Base Model:  McGill-NLP/AfriqueGemma-4B
LoRA Source: fallback-ai/Muria-Afrique-Gemma-4B-LoRA
Target Repo: fallback-ai/Muria-Afrique-Gemma-4B


## Merge LoRA Adapters into Base Model (16-Bit)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"Loading base tokenizer: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print(f"Loading base model: {BASE_MODEL} on CPU for safe weight merging...")
# Loading on CPU avoids GPU VRAM allocation pressure when loading full float16 weights
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True
)

print(f"Loading LoRA adapter from {LORA_REPO}...")
model = PeftModel.from_pretrained(base_model, LORA_REPO)

print("Merging LoRA adapter weights into base model...")
merged_model = model.merge_and_unload()

print(f"Saving merged standalone model to: {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

# Free memory
del base_model
del model
del merged_model
gc.collect()
print("Merged model successfully saved.")


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading base tokenizer: McGill-NLP/AfriqueGemma-4B...


config.json:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

...

adapter_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  262MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Merging LoRA adapter weights into base model...
Saving merged standalone model to: /kaggle/working/muria-afriquegemma-4b-faw-merged...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Setup llama.cpp & Convert to F16 GGUF

In [7]:
# Clone llama.cpp repository and fetch prebuilt release binaries
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp
!wget -q "https://github.com/ggml-org/llama.cpp/releases/download/b10107/llama-b10107-bin-ubuntu-x64.tar.gz" -O /kaggle/working/llama-release.tar.gz
!mkdir -p /kaggle/working/llama-release
!tar -xzf /kaggle/working/llama-release.tar.gz -C /kaggle/working/llama-release

LLAMA_BIN_DIR = "/kaggle/working/llama-release/llama-b10107"
!chmod +x {LLAMA_BIN_DIR}/llama-quantize {LLAMA_BIN_DIR}/llama-cli

F16_GGUF = f"{GGUF_DIR}/muria-afriquegemma-4b-faw-f16.gguf"
print(f"Converting {MERGED_DIR} to F16 GGUF -> {F16_GGUF}...")


!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {F16_GGUF} \
    --outtype f16


fatal: destination path '/kaggle/working/llama.cpp' already exists and is not an empty directory.
Converting /kaggle/working/muria-afriquegemma-4b-faw-merged to F16 GGUF -> /kaggle/working/muria_gguf_export/muria-afriquegemma-4b-faw-f16.gguf...
INFO:hf-to-gguf:Loading model: muria-afriquegemma-4b-faw-merged
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: Gemma3ForConditionalGeneration
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.float16 --> F16, shape = {2560, 262208}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.float16 --> F32, shape = {2560}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.float16 --> F16, shape = {10240, 2560}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.float16 --> F16, shape = {2560, 10240}
INFO:hf-to-gguf:blk.0.ffn_up

## Quantize to Q4_K_M for Android Mobile Edge Deployment

In [8]:
Q4_GGUF = f"{GGUF_DIR}/muria-afriquegemma-4b-faw-q4_k_m.gguf"
print(f"Quantizing F16 GGUF to Q4_K_M -> {Q4_GGUF}...")

!{LLAMA_BIN_DIR}/llama-quantize {F16_GGUF} {Q4_GGUF} Q4_K_M
!ls -lh {F16_GGUF} {Q4_GGUF}


Quantizing F16 GGUF to Q4_K_M -> /kaggle/working/muria_gguf_export/muria-afriquegemma-4b-faw-q4_k_m.gguf...
load_backend: loaded RPC backend from /kaggle/working/llama-release/llama-b10107/libggml-rpc.so
load_backend: loaded CPU backend from /kaggle/working/llama-release/llama-b10107/libggml-cpu-skylakex.so
llama_print_build_info: build = 10107 (c0bc8591e)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/kaggle/working/muria_gguf_export/muria-afriquegemma-4b-faw-f16.gguf' to '/kaggle/working/muria_gguf_export/muria-afriquegemma-4b-faw-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 37 key-value pairs and 444 tensors from /kaggle/working/muria_gguf_export/muria-afriquegemma-4b-faw-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma3
llama_model_loader

## Quick Verification Inference Test

In [9]:
# Run sample test using Gemma turn format
test_prompt = (
    "<start_of_turn>user\n"
    "What pesticide dosage should I mix in my knapsack sprayer for Fall Armyworm?<end_of_turn>\n"
    "<start_of_turn>model\n"
)

!{LLAMA_BIN_DIR}/llama-cli -m {Q4_GGUF} \
    -p "{test_prompt}" \
    -n 150




Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b10107-c0bc8591e
model      : /kaggle/working/muria_gguf_export/muria-afriquegemma-4b-faw-q4_k_m.gguf
ftype      : Q4_K - Medium
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> <start_of_turn>user
What pesticide dosage should I mix in my knapsack sprayer for Fall Armyworm?<end_of_turn>
<start_of_turn>model

_assistant
I need the active ingredient and product name before I can give a dosage. The label will tell you how much to mix per litre of water based on the product you are using. Sourc

## Publish GGUF Quantized Models & Model Card to Hugging Face

In [11]:
api.create_repo(repo_id=TARGET_REPO, repo_type="model", exist_ok=True, private=True)

# Upload Q4_K_M GGUF (primary mobile inference asset-option)
if os.path.exists(Q4_GGUF):
    size_gb = os.path.getsize(Q4_GGUF) / (1024 ** 3)
    print(f"Uploading {Q4_GGUF} ({size_gb:.2f} GB) to {TARGET_REPO}...")
    api.upload_file(
        path_or_fileobj=Q4_GGUF,
        path_in_repo="muria-afriquegemma-4b-faw-q4_k_m.gguf",
        repo_id=TARGET_REPO,
        repo_type="model"
    )
    print("Q4_K_M GGUF uploaded successfully.")

# Upload F16 GGUF
if os.path.exists(F16_GGUF):
    size_gb = os.path.getsize(F16_GGUF) / (1024 ** 3)
    print(f"Uploading {F16_GGUF} ({size_gb:.2f} GB) to {TARGET_REPO}...")
    api.upload_file(
        path_or_fileobj=F16_GGUF,
        path_in_repo="muria-afriquegemma-4b-faw-f16.gguf",
        repo_id=TARGET_REPO,
        repo_type="model"
    )
    print("F16 GGUF uploaded successfully.")

# Upload MODEL_CARD.md if found
for card_path in ["/kaggle/input/datasets/jodinho/model-card/MODEL_CARD.md", "MODEL_CARD.md"]:
    if os.path.exists(card_path):
        print(f"Uploading model card from {card_path} as README.md...")
        api.upload_file(
            path_or_fileobj=card_path,
            path_in_repo="README.md",
            repo_id=TARGET_REPO,
            repo_type="model"
        )
        print("README.md uploaded successfully.")
        break

print(f"\nAll exports complete: https://huggingface.co/{TARGET_REPO}")


Uploading /kaggle/working/muria_gguf_export/muria-afriquegemma-4b-faw-q4_k_m.gguf (2.33 GB) to fallback-ai/Muria-Afrique-Gemma-4B...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


Q4_K_M GGUF uploaded successfully.
Uploading /kaggle/working/muria_gguf_export/muria-afriquegemma-4b-faw-f16.gguf (7.24 GB) to fallback-ai/Muria-Afrique-Gemma-4B...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

F16 GGUF uploaded successfully.
Uploading model card from /kaggle/input/datasets/jodinho/model-card/MODEL_CARD.md as README.md...
README.md uploaded successfully.

All exports complete: https://huggingface.co/fallback-ai/Muria-Afrique-Gemma-4B


In [1]:
# CONVERT TO CACTUS (.cact) BINARY & PUBLISH TO HUGGING FACE
import os, sys, glob, subprocess, shutil, json as _json, torch
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from safetensors.torch import save_file

print("--- Starting Cactus (.cact) Export Pipeline ---")

WORKING_DIR    = "/kaggle/working"
CACTUS_SRC_DIR = f"{WORKING_DIR}/cactus_src"
CACT_EXPORT_DIR= f"{WORKING_DIR}/muria_cact_export"
CLEAN_LM_DIR   = f"{WORKING_DIR}/muria_clean_lm"
BASE_MODEL     = "McGill-NLP/AfriqueGemma-4B"
LORA_ADAPTER   = "fallback-ai/Muria-Afrique-Gemma-4B-LoRA"
TARGET_HF_REPO = "fallback-ai/Muria-Afrique-Gemma-4B"

for d in [CACT_EXPORT_DIR, CLEAN_LM_DIR]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)
for old in [f"{WORKING_DIR}/cactus"]:
    if os.path.exists(old): shutil.rmtree(old, ignore_errors=True)

# ── Step 1: Upgrade torchao + peft ───────────────────────────────────────────
print("\nStep 1/5: Upgrading torchao + peft...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao", "peft"], check=True)
print("  Done.")

# ── Step 2: Auth ──────────────────────────────────────────────────────────────
print("\nStep 2/5: Authenticating...")
try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN2")
except Exception:
    hf_token = os.environ.get("HF_TOKEN2") or os.environ.get("HF_TOKEN")
assert hf_token, "HF_TOKEN2 not found!"
login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)
print("  Authenticated.")

# ── Step 3: LoRA ──────────────────────────────────────────────────────────────
print("\nStep 3/5: LoRA adapter...")
local_lora_dir = f"{WORKING_DIR}/muria_lora_local"
if not os.path.exists(local_lora_dir) or len(os.listdir(local_lora_dir)) == 0:
    print(f"  Downloading to {local_lora_dir}...")
    snapshot_download(repo_id=LORA_ADAPTER, local_dir=local_lora_dir, token=hf_token)
else:
    print(f"  Already present at {local_lora_dir}.")

# ── Step 4: Merge LoRA + CORRECT key remapping ────────────────────────────────
print("\nStep 4/5: Merge LoRA + remap tensor keys for Cactus compatibility...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token)
tokenizer.save_pretrained(CLEAN_LM_DIR)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.float16, device_map="auto",
    token=hf_token, trust_remote_code=True,
)
print(f"  Loaded: {type(model).__name__}")

# Show actual top-level key prefixes so we can verify mapping
sd_sample = list(model.state_dict().keys())
top_prefixes = sorted(set(k.split('.')[0] for k in sd_sample))
print(f"  Top-level state_dict prefixes: {top_prefixes}")

model = PeftModel.from_pretrained(model, local_lora_dir)
model = model.merge_and_unload()
print("  LoRA merged.")

# KEY REMAP (corrected):
# Actual state_dict prefixes after merge are 'model.*' and 'lm_head.*'
# The VLM structure is: model.language_model.layers.X.* (LM body)
#                        model.vision_tower.*              (skip)
#                        model.multi_modal_projector.*     (skip)
#                        lm_head.*                         (keep)
# Cactus GemmaForCausalLM expects: model.layers.X.*, model.embed_tokens.*, model.norm.*, lm_head.*
# So we strip 'model.language_model.' -> 'model.' and keep 'lm_head.*'
print("  Remapping state dict keys...")
state_dict = model.state_dict()
top_prefixes_merged = sorted(set(k.split('.')[0] + '.' + k.split('.')[1] if len(k.split('.')) > 1 else k.split('.')[0] for k in list(state_dict.keys())[:20]))
print(f"  Sample prefixes (merged): {top_prefixes_merged[:8]}")

lm_dict = {}
skipped = set()
for k, v in state_dict.items():
    if k.startswith("model.language_model."):
        # 'model.language_model.layers.0.self_attn.q_proj.weight'
        # → 'model.layers.0.self_attn.q_proj.weight'
        new_k = "model." + k[len("model.language_model."):]
        lm_dict[new_k] = v.contiguous()
    elif k.startswith("lm_head."):
        lm_dict[k] = v.contiguous()
    else:
        skipped.add(k.split(".")[0] + "." + k.split(".")[1] if "." in k else k)

print(f"  Extracted {len(lm_dict)} LM tensors.")
print(f"  Skipped (vision/projector): {len(state_dict) - len(lm_dict)} tensors")
if lm_dict:
    print(f"  Sample keys: {list(lm_dict.keys())[:3]}")
else:
    # FALLBACK: if language_model prefix wasn't found, dump actual prefixes and try alternatives
    all_prefixes = set('.'.join(k.split('.')[:3]) for k in state_dict.keys())
    print(f"  WARNING: 0 extracted! Actual 3-level prefixes: {sorted(all_prefixes)[:10]}")
    # Try keeping everything that looks like LM (not vision_tower, not multi_modal_projector)
    for k, v in state_dict.items():
        if not any(x in k for x in ["vision_tower", "multi_modal_projector", "vision_model"]):
            lm_dict[k] = v.contiguous()
    print(f"  Fallback extracted {len(lm_dict)} tensors.")

# Save remapped weights
save_file(lm_dict, os.path.join(CLEAN_LM_DIR, "model.safetensors"))
print(f"  model.safetensors saved ({os.path.getsize(os.path.join(CLEAN_LM_DIR,'model.safetensors'))/1024**3:.2f} GB)")

# Save text-only config
raw_cfg = model.config
text_cfg_dict = (raw_cfg.text_config.to_dict()
                 if hasattr(raw_cfg, "text_config") else raw_cfg.to_dict())
for drop in ["vision_config", "projector_config"]:
    text_cfg_dict.pop(drop, None)
text_cfg_dict["architectures"] = ["GemmaForCausalLM"]
text_cfg_dict["model_type"] = "gemma"
text_cfg_dict["tie_word_embeddings"] = False
with open(os.path.join(CLEAN_LM_DIR, "config.json"), "w") as f:
    _json.dump(text_cfg_dict, f, indent=2)
print(f"  config.json: layers={text_cfg_dict.get('num_hidden_layers')}")

del model, state_dict, lm_dict
torch.cuda.empty_cache()

# ── Step 5: Clone Cactus + build libcactus_engine.so from source ──────────────
# pip install cactus-compute installs ONLY the Python CLI (1.3MB).
# The C++ engine (libcactus_engine.so) is NOT in the PyPI wheel — must be compiled.
print("\nStep 5/5: Setting up Cactus + building C++ engine...")
if not os.path.exists(CACTUS_SRC_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/cactus-compute/cactus", CACTUS_SRC_DIR], check=True)
else:
    print(f"  Cactus source already at {CACTUS_SRC_DIR}.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                f"{CACTUS_SRC_DIR}/python"], check=True)

cli_bin = shutil.which("cactus") or "cactus"
print(f"  CLI: {cli_bin}")

# Build libcactus_engine.so — this compiles the Caesar C++ generator
print("  Building libcactus_engine.so (takes ~5 min)...")
build_res = subprocess.run([cli_bin, "build", "--python"], cwd=CACTUS_SRC_DIR)
if build_res.returncode != 0:
    print("  cactus build failed, trying cmake directly...")
    subprocess.run(["cmake", "-B", "build", "-DCMAKE_BUILD_TYPE=Release"],
                   cwd=CACTUS_SRC_DIR, check=True)
    subprocess.run(["cmake", "--build", "build", f"-j{os.cpu_count() or 4}",
                    "--target", "cactus_python"], cwd=CACTUS_SRC_DIR, check=True)

# Confirm .so exists
so_files = glob.glob(f"{CACTUS_SRC_DIR}/**/*.so", recursive=True)
print(f"  .so files built: {[os.path.basename(f) for f in so_files]}")

# ── Convert ───────────────────────────────────────────────────────────────────
env = {**os.environ,
       "PYTHONPATH": f"{CACTUS_SRC_DIR}/python:" + os.environ.get("PYTHONPATH", ""),
       "HF_TOKEN": hf_token, "HUGGING_FACE_HUB_TOKEN": hf_token}

convert_cmd = [cli_bin, "convert", CLEAN_LM_DIR, CACT_EXPORT_DIR,
               "--bits", "4", "--token", hf_token]
safe_cmd = [c if not c.startswith("hf_") else "hf_***" for c in convert_cmd]
print("  Command:", " ".join(safe_cmd))

res = subprocess.run(convert_cmd, env=env)
if res.returncode != 0:
    print(f"  CLI exited {res.returncode}. Retrying with python -m cactus...")
    subprocess.run([sys.executable, "-m", "cactus", "convert",
                    CLEAN_LM_DIR, CACT_EXPORT_DIR, "--bits", "4", "--token", hf_token],
                   check=True, env=env)
print("\nCactus conversion completed!")

# ── Find .cact + upload ───────────────────────────────────────────────────────
found = list(set(
    f for pat in [f"{CACT_EXPORT_DIR}/**/*.cact", f"{CACT_EXPORT_DIR}/*.cact",
                  f"{WORKING_DIR}/**/*.cact"]
    for f in glob.glob(pat, recursive=True)
))
assert found, f"No .cact found in {CACT_EXPORT_DIR}"
cact_file   = found[0]
target_name = "muria-afriquegemma-4b-faw.cact"
print(f"\nFOUND: {cact_file}  ({os.path.getsize(cact_file)/1024**3:.2f} GB)")

print(f"\nUploading to https://huggingface.co/{TARGET_HF_REPO}...")
api.create_repo(repo_id=TARGET_HF_REPO, repo_type="model", exist_ok=True, private=True)
api.upload_file(path_or_fileobj=cact_file, path_in_repo=target_name,            repo_id=TARGET_HF_REPO, repo_type="model")
api.upload_file(path_or_fileobj=cact_file, path_in_repo=f"cact/{target_name}",  repo_id=TARGET_HF_REPO, repo_type="model")
print(f"\nUpload complete: https://huggingface.co/{TARGET_HF_REPO}")


--- Starting Cactus (.cact) Export Pipeline ---

Step 1/5: Upgrading torchao + peft...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 21.0 MB/s eta 0:00:00
  Done.

Step 2/5: Authenticating...
  Authenticated.

Step 3/5: LoRA adapter...


Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]


Step 4/5: Merge LoRA + remap tensor keys for Cactus compatibility...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

  LoRA merged.
  Remapping state dict keys...
  Sample prefixes (merged): ['model.vision_tower']
  Extracted 445 LM tensors.
  Skipped (vision/projector): 439 tensors
  Sample keys: ['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.k_proj.weight']
  model.safetensors saved (8.48 GB)
  config.json: layers=34

Step 5/5: Setting up Cactus + building C++ engine...


Cloning into '/kaggle/working/cactus_src'...


  CLI: /usr/local/bin/cactus
  Building libcactus_engine.so (takes ~5 min)...
Building cactus-engine...
Building Cactus for Python...
Build failed
  cactus build failed, trying cmake directly...


cc1plus: error: bad value (‘armv8.2-a+fp16+simd+dotprod+i8mm’) for ‘-march=’ switch
cc1plus: error: bad value (‘armv8.2-a+fp16+simd+dotprod+i8mm’) for ‘-march=’ switch
cc1plus: error: bad value (‘armv8.2-a+fp16+simd+dotprod+i8mm’) for ‘-march=’ switch
cc1plus: error: bad value (‘armv8.2-a+fp16+simd+dotprod+i8mm’) for ‘-march=’ switch
cc1plus: note: valid arguments to ‘-march=’ switch are: nocona core2 nehalem corei7 westmere sandybridge corei7-avx ivybridge core-avx-i haswell core-avx2 broadwell skylake skylake-avx512 cannonlake icelake-client rocketlake icelake-server cascadelake tigerlake cooperlake sapphirerapids alderlake bonnell atom silvermont slm goldmont goldmont-plus tremont knl knm x86-64 x86-64-v2 x86-64-v3 x86-64-v4 eden-x2 nano nano-1000 nano-2000 nano-3000 nano-x2 eden-x4 nano-x4 k8 k8-sse3 opteron opteron-sse3 athlon64 athlon64-sse3 athlon-fx amdfam10 barcelona bdver1 bdver2 bdver3 bdver4 znver1 znver2 znver3 btver1 btver2 native
cc1plus: note: valid arguments to ‘-march

CalledProcessError: Command '['cmake', '-B', 'build', '-DCMAKE_BUILD_TYPE=Release']' returned non-zero exit status 1.

In [2]:
import os, glob
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN2")
login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)

CLEAN_LM_DIR = "/kaggle/working/muria_clean_lm"
TARGET_REPO  = "fallback-ai/Muria-Afrique-Gemma-4B"

api.create_repo(repo_id=TARGET_REPO, repo_type="model", exist_ok=True, private=True)

for fname in os.listdir(CLEAN_LM_DIR):
    fpath = os.path.join(CLEAN_LM_DIR, fname)
    if not os.path.isfile(fpath):
        continue
    repo_path = f"clean-checkpoint/{fname}"
    size_mb = os.path.getsize(fpath) / 1024**2
    print(f"Uploading {fname} ({size_mb:.1f} MB) → {repo_path}")
    api.upload_file(
        path_or_fileobj=fpath,
        path_in_repo=repo_path,
        repo_id=TARGET_REPO,
        repo_type="model",
    )

print(f"\nDone: https://huggingface.co/{TARGET_REPO}/tree/main/clean-checkpoint")


Uploading tokenizer_config.json (0.0 MB) → clean-checkpoint/tokenizer_config.json
Uploading chat_template.jinja (0.0 MB) → clean-checkpoint/chat_template.jinja
Uploading model.safetensors (8681.4 MB) → clean-checkpoint/model.safetensors


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading config.json (0.0 MB) → clean-checkpoint/config.json
Uploading tokenizer.json (31.8 MB) → clean-checkpoint/tokenizer.json


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Done: https://huggingface.co/fallback-ai/Muria-Afrique-Gemma-4B/tree/main/clean-checkpoint
